# Comprehensive v3 external visual evaluation

This evaluates the saved SDFVD 2.0 five-fold visual checkpoints on one median-index face crop per filename-derived video stem. It does not retrain. The report remains provisional because the observed archive counts and several filename-to-source mappings are unresolved. AUC and accuracy are candidate-video weighted; related candidates may share source videos.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import shutil, zipfile, subprocess, sys, json, time
BASE = Path('/content/drive/MyDrive/Datasets')
DATASET_ARCHIVE = BASE / 'Comprehensive_Deepfake_Dataset_v3.zip'
CV_ROOT = BASE / 'REPLACE_WITH_YOUR_SDFVD2_CV_RUN_FOLDER' / 'group_cv'
SOURCE_B64 = BASE / 'poor_guys_deepfake_colab_source.b64'
assert DATASET_ARCHIVE.is_file(), f'Missing dataset archive: {DATASET_ARCHIVE}'
assert CV_ROOT.is_dir(), f'Missing SDFVD2 CV checkpoints: {CV_ROOT}'
assert SOURCE_B64.is_file(), f'Missing source bundle: {SOURCE_B64}'
outer_size = DATASET_ARCHIVE.stat().st_size
if shutil.disk_usage('/content').free < outer_size * 2: raise RuntimeError('Not enough Colab temporary disk for the archive copy and nested ZIP. Check runtime storage before proceeding.')
print('Input paths found; dataset archive bytes:', outer_size, '| Colab free bytes:', shutil.disk_usage('/content').free)

In [ ]:
# Copy/extract the nested archive to temporary Colab storage. This can take several minutes and needs space for both ZIP layers plus extracted frames.
WORK = Path('/content/comprehensive_external_eval')
OUTER_COPY = Path('/content/comprehensive_v3_wrapper.zip')
INNER_ZIP = Path('/content/comprehensive_v3_inner.zip')
FRAME_ROOT = WORK / 'dataset'
SOURCE_ROOT = WORK / 'source'
SOURCE_ZIP = Path('/content/poor_guys_deepfake_colab_source.zip')
WORK.mkdir(parents=True, exist_ok=True)
COMPLETE = WORK / 'dataset_extraction_complete'
if COMPLETE.is_file():
    print('Reusing completed dataset extraction:', FRAME_ROOT)
else:
    shutil.copyfile(DATASET_ARCHIVE, OUTER_COPY)
    with zipfile.ZipFile(OUTER_COPY) as outer:
        inner_names = [n for n in outer.namelist() if n.lower().endswith('.zip') and not n.startswith('__MACOSX/')]
        if len(inner_names) != 1: raise RuntimeError(f'Expected one nested dataset ZIP; found {inner_names[:5]}')
        with outer.open(inner_names[0]) as src, INNER_ZIP.open('wb') as dst: shutil.copyfileobj(src, dst, length=8*1024*1024)
    with zipfile.ZipFile(INNER_ZIP) as inner:
        needed = sum(info.file_size for info in inner.infolist())
        free = shutil.disk_usage('/content').free
        if free < needed + 1024**3: raise RuntimeError(f'Not enough Colab free disk to extract archive safely: need about {needed + 1024**3} bytes, have {free}')
        FRAME_ROOT.mkdir(parents=True, exist_ok=True)
        inner.extractall(FRAME_ROOT)  # Reading each member to EOF also checks its ZIP CRC.
    COMPLETE.write_text('ok', encoding='utf-8')
SOURCE_ROOT.mkdir(parents=True, exist_ok=True)
import base64
SOURCE_ZIP.write_bytes(base64.b64decode(SOURCE_B64.read_bytes().strip()))
with zipfile.ZipFile(SOURCE_ZIP) as src: src.extractall(SOURCE_ROOT)
PROJECT = SOURCE_ROOT / 'poor_guys_deepfake_project'
print('Extracted dataset and pipeline source to temporary Colab storage:', FRAME_ROOT, PROJECT)

In [ ]:
# Run the saved visual branch checkpoints. Do not set --mapping-reviewed unless the full mapping and archive counts have been independently reconciled.
IMAGE_ROOT = FRAME_ROOT / 'deepfake_dataset'
OUTPUT = BASE / f'comprehensive_external_eval_{time.strftime("%Y%m%dT%H%M%S", time.gmtime())}'
cmd = [sys.executable, str(PROJECT / 'multimodal' / 'evaluate_external_frames.py'), '--frames-root', str(IMAGE_ROOT), '--cv-root', str(CV_ROOT), '--output-dir', str(OUTPUT), '--device', 'auto']
print('Starting external evaluation...')
subprocess.run(cmd, cwd=PROJECT, check=True)
print('Report:', OUTPUT / 'external_evaluation.json')
print('Predictions:', OUTPUT / 'external_predictions.csv')

## Interpretation

Use the fold mean and spread as an exploratory external-domain result. The score does not establish source-independent performance because filename-derived candidates may share source videos, and the archive count discrepancy is still unresolved. This dataset contains pre-cropped face images, while SDFVD2 training used frames extracted from videos, so the input-domain shift is part of what this evaluation measures. The script uses each fold's SDFVD2 validation threshold and never calibrates against Comprehensive v3.